# Hybrid Search Bm25 Plus Sbert

**Notebook:** `05-hybrid-search-bm25-plus-sbert.ipynb`

This notebook is part of the Personal Knowledge Engine project.

In [ ]:
# # Start coding here
# !pip install rank_bm25

  Using cached rank_bm25-0.2.2-py3-none-any.whl.metadata (3.2 kB)
Using cached rank_bm25-0.2.2-py3-none-any.whl (8.6 kB)



[notice] A new release of pip is available: 24.0 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [2]:
# ==========================================================
# Notebook 05: Hybrid Search (BM25 + SBERT)
# ==========================================================

import faiss
import pickle
import numpy as np
import pandas as pd

from rank_bm25 import BM25Okapi

from sentence_transformers import SentenceTransformer

d:\Books\projects-nlp-transformers-learning\.projectnlps\Lib\site-packages\sentence_transformers\cross_encoder\CrossEncoder.py:11: TqdmExperimentalWarning: Using `tqdm.autonotebook.tqdm` in notebook mode. Use `tqdm.tqdm` instead to force console mode (e.g. in jupyter console)
  from tqdm.autonotebook import tqdm, trange


In [3]:
chunks_df = pd.read_csv("data/processed/chunked_corpus.csv")

with open("data/embeddings/chunk_embeddings.pkl", "rb") as file:

    embeddings = pickle.load(file)

faiss_index = faiss.read_index("data/embeddings/faiss_index.bin")

print(chunks_df.shape)

(2865, 5)


In [4]:
model = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2")

d:\Books\projects-nlp-transformers-learning\.projectnlps\Lib\site-packages\huggingface_hub\file_download.py:949: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(


In [5]:
tokenized_corpus = [text.lower().split() for text in chunks_df["chunk_text"].tolist()]

In [6]:
bm25 = BM25Okapi(tokenized_corpus)

print("BM25 Index Built Successfully.")

BM25 Index Built Successfully.


In [7]:
query = "Docker Desktop"

tokenized_query = query.lower().split()

bm25_scores = bm25.get_scores(tokenized_query)

bm25_scores[:5]

array([0., 0., 0., 0., 0.])

In [8]:
bm25_results = chunks_df.copy()

bm25_results["bm25_score"] = bm25_scores

bm25_results.sort_values(by="bm25_score", ascending=False).head(3)

,chunk_id,source,category,created_date,chunk_text,bm25_score
2187,2188,book.pdf,book,NaN,"taset, this is a very common situation. In our...",6.948131
0,1,2026-06-14.md,journal,2026-06-14,# Daily Journal\n\nToday I read about local AI...,0.000000
1913,1914,book.pdf,book,NaN,t needing any\nlabels at all! The key idea is ...,0.000000


In [9]:
query_embedding = model.encode(query)

query_embedding = np.array([query_embedding]).astype("float32")

faiss.normalize_L2(query_embedding)

In [10]:
top_k = len(chunks_df)

semantic_scores, semantic_indices = faiss_index.search(query_embedding, top_k)

In [11]:
semantic_score_array = np.zeros(len(chunks_df))

for score, idx in zip(semantic_scores[0], semantic_indices[0]):

    semantic_score_array[idx] = score

In [12]:
semantic_results = chunks_df.copy()

semantic_results["semantic_score"] = semantic_score_array

semantic_results.sort_values(by="semantic_score", ascending=False).head(3)

,chunk_id,source,category,created_date,chunk_text,semantic_score
1367,1368,book.pdf,book,NaN,"cluster_uuid"" : ""ABGDdvbbRWmMb9Umz79HbA"",\n""ve...",0.233787
942,943,book.pdf,book,NaN,panese.\nInteracting with Model Widgets\nIn th...,0.224609
1366,1367,book.pdf,book,NaN,ies the ID of the subprocess we wish to use. B...,0.223629


In [13]:
def min_max_normalize(values):

    values = np.array(values)

    return (values - values.min()) / (values.max() - values.min() + 1e-9)

In [14]:
bm25_normalized = min_max_normalize(bm25_scores)

semantic_normalized = min_max_normalize(semantic_score_array)

In [15]:
ALPHA = 0.4

hybrid_scores = ALPHA * bm25_normalized + (1 - ALPHA) * semantic_normalized

In [16]:
hybrid_results = chunks_df.copy()

hybrid_results["bm25_score"] = bm25_normalized

hybrid_results["semantic_score"] = semantic_normalized

hybrid_results["hybrid_score"] = hybrid_scores

In [17]:
top_results = hybrid_results.sort_values(by="hybrid_score", ascending=False).head(5)

top_results[["source", "bm25_score", "semantic_score", "hybrid_score", "chunk_text"]]

,source,bm25_score,semantic_score,hybrid_score,chunk_text
2187,book.pdf,1.0,0.572399,0.743440,"taset, this is a very common situation. In our..."
1367,book.pdf,0.0,1.000000,0.600000,"cluster_uuid"" : ""ABGDdvbbRWmMb9Umz79HbA"",\n""ve..."
942,book.pdf,0.0,0.979390,0.587634,panese.\nInteracting with Model Widgets\nIn th...
1366,book.pdf,0.0,0.977188,0.586313,ies the ID of the subprocess we wish to use. B...
193,book.pdf,0.0,0.893284,0.535970,f the coolest features of the Hub is that you ...


In [18]:
def hybrid_search(query, dataframe, bm25, model, faiss_index, alpha=0.4, top_k=5):

    # -----------------------------
    # BM25
    # -----------------------------
    tokenized_query = query.lower().split()

    bm25_scores = bm25.get_scores(tokenized_query)

    # -----------------------------
    # Semantic Search
    # -----------------------------
    query_embedding = model.encode(query)

    query_embedding = np.array([query_embedding]).astype("float32")

    faiss.normalize_L2(query_embedding)

    semantic_scores, semantic_indices = faiss_index.search(
        query_embedding, len(dataframe)
    )

    semantic_score_array = np.zeros(len(dataframe))

    for score, idx in zip(semantic_scores[0], semantic_indices[0]):

        semantic_score_array[idx] = score

    # -----------------------------
    # Normalize
    # -----------------------------
    bm25_norm = min_max_normalize(bm25_scores)

    semantic_norm = min_max_normalize(semantic_score_array)

    # -----------------------------
    # Hybrid Fusion
    # -----------------------------
    hybrid_score = alpha * bm25_norm + (1 - alpha) * semantic_norm

    results = dataframe.copy()

    results["bm25_score"] = bm25_norm

    results["semantic_score"] = semantic_norm

    results["hybrid_score"] = hybrid_score

    return results.sort_values(by="hybrid_score", ascending=False).head(top_k)

In [19]:
results = hybrid_search(
    query="Tell me about Docker Desktop",
    dataframe=chunks_df,
    bm25=bm25,
    model=model,
    faiss_index=faiss_index,
    alpha=0.4,
    top_k=5,
)

results[["source", "hybrid_score", "chunk_text"]]

,source,hybrid_score,chunk_text
443,book.pdf,0.672554,t that the model does not improperly exploit c...
192,book.pdf,0.619675,let you reproduce published results or leverag...
2771,book.pdf,0.600000,"function, Error Analysis\nPaperspace Gradient..."
2514,book.pdf,0.587752,tarting with a model that is\nhundreds of giga...
2701,book.pdf,0.583449,"Custom Model, Choosing a Good Student Initial..."
